# HealthAI Disease Classifier — v3.0
## Bio_ClinicalBERT · 12 diseases · Equal Neuro + Vector-Borne Priority

### What is new in v3.0

**1. Reduced dengue inputs — dengue specificity gate**
Dengue now requires at least one clinical marker (NS1 antigen, platelet,
thrombocytopenia, leucopenia, retro-orbital) to receive a high-confidence
prediction. Generic inputs like `fever and headache` trigger
`DENGUE_GATE_PENALTY` which suppresses dengue probability so the next
most-plausible disease surfaces instead.

**2. Equal neurological priority**
The six neurological diseases (epilepsy, migraine, stroke, diabetic_neuropathy,
dementia, parkinsons) receive `NEURO_WEIGHT_BOOST = 1.30` in training loss
AND `NEURO_TFIDF_BOOST = 1.25` in the TF-IDF cosine blend — making their
effective priority equal to the six vector-borne diseases.

**3. Three-layer correction stack (built on TF-IDF + vector matrix theory)**
- Layer 1: BERT softmax — primary signal
- Layer 2: TF-IDF centroid cosine blend — triggered when uncertain / dengue-biased
- Layer 3: Dengue specificity gate — hard penalty when no clinical markers present

**Before running:** Runtime → Change runtime type → T4 GPU
**Upload dataset** to the Colab file browser before running.


## Step 1 — Install & imports


In [ ]:
!pip install transformers torch scikit-learn matplotlib seaborn --quiet
print('Libraries ready.')


## Step 2 — Configuration


In [ ]:
import os, json, random, platform
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, f1_score
)

# -- Paths
CSV_PATH   = 'healthai_synthetic_training_set_7200.csv'
JSONL_PATH = 'healthai_synthetic_training_set_7200.jsonl'

# -- Hyperparameters
MODEL_NAME    = 'emilyalsentzer/Bio_ClinicalBERT'
MAX_LEN       = 128
BATCH_SIZE    = 32
PHASE1_EPOCHS = 2
PHASE2_EPOCHS = 3
LR_HEAD       = 1e-3
LR_FULL       = 2e-5
PATIENCE      = 2
SEED          = 42

# -- Label map (neurological first, then vector-borne)
DISEASE_LABELS = {
    'epilepsy': 0, 'migraine': 1, 'stroke': 2, 'diabetic_neuropathy': 3,
    'dementia': 4, 'parkinsons': 5,
    'dengue': 6, 'malaria': 7, 'kala_azar': 8,
    'chikungunya': 9, 'japanese_encephalitis': 10, 'scrub_typhus': 11,
}
LABEL_NAMES = {v: k for k, v in DISEASE_LABELS.items()}
NUM_LABELS  = len(DISEASE_LABELS)

# -- Disease groups
NEURO_DISEASES  = [
    'epilepsy', 'migraine', 'stroke',
    'diabetic_neuropathy', 'dementia', 'parkinsons',
]
VECTOR_DISEASES = [
    'dengue', 'malaria', 'kala_azar',
    'chikungunya', 'japanese_encephalitis', 'scrub_typhus',
]

# -- Neurological priority boost
# Applied in (a) training loss weights  (b) TF-IDF blend scoring
NEURO_WEIGHT_BOOST = 1.30  # class-weight multiplier during training
NEURO_TFIDF_BOOST  = 1.25  # cosine score multiplier in ambiguous blending

# -- Dengue specificity gate
# Requires at least one clinical marker for a dengue prediction to stand.
# 'fever' and 'headache' are NOT markers -- they appear in all diseases.
DENGUE_MARKER_GATE      = True
DENGUE_GATE_PENALTY     = 0.20   # multiply dengue prob by this when no markers
DENGUE_SPECIFIC_MARKERS = [
    'ns1', 'ns1 antigen', 'platelet', 'thrombocytopenia', 'leucopenia',
    'retro-orbital', 'retro orbital', 'tourniquet', 'dengue fever',
]

# -- TF-IDF blend
TFIDF_BLEND_ALPHA   = 0.65  # BERT weight (1 - alpha goes to TF-IDF)
AMBIGUITY_THRESHOLD = 0.55  # blend when BERT top-prob < this
DENGUE_OVERRIDE_CAP = 0.72  # also blend when dengue pred < this

# -- Baseline
RANDOM_BASELINE_LOSS = float(np.log(NUM_LABELS))

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'HealthAI v3 config ready. {NUM_LABELS} classes.')
print(f'Neurological  : {NEURO_DISEASES}')
print(f'Vector-borne  : {VECTOR_DISEASES}')
print(f'NEURO_WEIGHT_BOOST  = {NEURO_WEIGHT_BOOST}')
print(f'NEURO_TFIDF_BOOST   = {NEURO_TFIDF_BOOST}')
print(f'DENGUE_MARKER_GATE  = {DENGUE_MARKER_GATE}')
print(f'DENGUE_GATE_PENALTY = {DENGUE_GATE_PENALTY}')
print(f'Random baseline loss = {RANDOM_BASELINE_LOSS:.3f}')


## Step 3 — GPU check


In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if str(DEVICE) != 'cuda':
    raise RuntimeError(
        'GPU not detected!\n'
        'Go to: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU\n'
        'Then: Runtime -> Restart and run all'
    )
torch.cuda.manual_seed_all(SEED)
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


## Step 4 — Load & validate dataset


In [ ]:
if os.path.exists(CSV_PATH):
    df_raw = pd.read_csv(CSV_PATH)
    print(f'Loaded CSV: {len(df_raw)} rows')
elif os.path.exists(JSONL_PATH):
    df_raw = pd.DataFrame([json.loads(l) for l in open(JSONL_PATH)])
    print(f'Loaded JSONL: {len(df_raw)} rows')
else:
    raise FileNotFoundError(
        f'Neither {CSV_PATH} nor {JSONL_PATH} found.\n'
        'Upload the dataset to the Colab file browser and re-run.'
    )

assert 'label' in df_raw.columns and 'input_text' in df_raw.columns
df_raw = df_raw[df_raw['label'].isin(DISEASE_LABELS)].copy()
df_raw['label_id']   = df_raw['label'].map(DISEASE_LABELS)
df_raw['input_text'] = df_raw['input_text'].astype(str).str.strip()

print(f'Valid samples : {len(df_raw)}')
print(f'Classes       : {df_raw["label"].nunique()}')
print()
vc = df_raw['label'].value_counts()
print('Samples per class:')
for d in NEURO_DISEASES + VECTOR_DISEASES:
    tag = '[neuro ]' if d in NEURO_DISEASES else '[vector]'
    print(f'  {tag} {d:25s}: {vc.get(d, 0)}')
assert df_raw['label'].nunique() == NUM_LABELS, 'Missing disease classes!'
print('\nData validated.')


## Step 5 — Train / val / test split (70 / 15 / 15)


In [ ]:
train_df, temp_df = train_test_split(
    df_raw, test_size=0.30, random_state=SEED, stratify=df_raw['label_id'])
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['label_id'])

print(f'Train: {len(train_df):5d}  Val: {len(val_df):5d}  Test: {len(test_df):5d}')
print()
print('Train class distribution:')
print(train_df['label'].value_counts().to_string())


## Step 5b — TF-IDF Disease Profiler

Builds a **per-disease TF-IDF centroid** (mean TF-IDF vector across all training
samples of that disease) from the full document-term matrix.

**v3 additions over v2:**
1. `score(boost_neuro=True)` multiplies neurological cosine scores by
   `NEURO_TFIDF_BOOST` before normalisation — equalising neuro vs vector priority.
2. TF-IDF is fitted on training split only (zero data leakage).


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as _cosine_sim


class TFIDFDiseaseProfiler:
    """
    Sparse vector-space disease profiler (document-term matrix).

    Each disease = centroid (mean TF-IDF vector) of all its training records.
    High-IDF words (disease-specific markers) dominate each centroid.
    Generic words (fever, headache) have near-zero IDF -> near-zero centroid weight.

    At inference:
      query text -> TF-IDF vector -> cosine similarity vs all 12 centroids
      -> normalised probability distribution over diseases
    """

    def __init__(self, max_features=8000, ngram_range=(1, 2)):
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=ngram_range,
            sublinear_tf=True,   # 1 + log(TF): dampens very frequent terms
            min_df=2,            # ignore terms in < 2 documents
            stop_words='english',
        )
        self.centroids     = {}   # {label_id: np.ndarray (1, V)}
        self.feature_names = None

    def fit(self, df):
        """Fit on training split ONLY — no val/test leakage."""
        texts        = df['input_text'].tolist()
        tfidf_matrix = self.vectorizer.fit_transform(texts)    # sparse (N, V)
        self.feature_names = self.vectorizer.get_feature_names_out()
        for label_id in range(NUM_LABELS):
            mask     = (df['label_id'].values == label_id)
            centroid = np.asarray(tfidf_matrix[mask].mean(axis=0))  # (1, V)
            self.centroids[label_id] = centroid
        V = len(self.feature_names)
        print(f'TF-IDF profiler fitted | vocab={V:,} | {NUM_LABELS} centroids')
        print()
        print('Top-5 signature terms per disease:')
        for lid in range(NUM_LABELS):
            top_idx   = np.argsort(self.centroids[lid][0])[-5:][::-1]
            top_terms = [self.feature_names[i] for i in top_idx]
            tag = '[neuro ]' if LABEL_NAMES[lid] in NEURO_DISEASES else '[vector]'
            print(f'  {tag} {LABEL_NAMES[lid]:25s}: {", ".join(top_terms)}')

    def score(self, text, boost_neuro=False):
        """
        Cosine similarity of query text against every disease centroid.

        Parameters
        ----------
        text        : patient symptom string
        boost_neuro : if True, multiply neurological scores by NEURO_TFIDF_BOOST
                      before normalisation — used in ambiguous-case blending.

        Returns
        -------
        sims : np.ndarray shape (NUM_LABELS,), normalised to sum = 1.
        """
        vec  = self.vectorizer.transform([text])           # sparse (1, V)
        sims = np.zeros(NUM_LABELS, dtype=np.float64)
        for lid, centroid in self.centroids.items():
            sims[lid] = float(_cosine_sim(vec, centroid)[0, 0])
        if boost_neuro:
            for name, lid in DISEASE_LABELS.items():
                if name in NEURO_DISEASES:
                    sims[lid] *= NEURO_TFIDF_BOOST
        total = sims.sum()
        if total > 1e-10:
            sims /= total
        return sims


tfidf_profiler = TFIDFDiseaseProfiler(max_features=8000, ngram_range=(1, 2))
tfidf_profiler.fit(train_df)
print()
print('TF-IDF profiler ready.')


## Step 6 — Tokenize & DataLoaders


In [ ]:
print(f'Loading tokenizer from {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class DiseaseDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts  = list(texts)
        self.labels = list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = tokenizer(
            str(self.texts[i]),
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[i], dtype=torch.long),
        }


_nw = 0 if platform.system() == 'Windows' else 2

train_ds = DiseaseDataset(train_df['input_text'], train_df['label_id'])
val_ds   = DiseaseDataset(val_df['input_text'],   val_df['label_id'])
test_ds  = DiseaseDataset(test_df['input_text'],  test_df['label_id'])

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=_nw)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=_nw)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=_nw)

print(f'Train batches: {len(train_dl)}  Val: {len(val_dl)}  Test: {len(test_dl)}')
print(f'MAX_LEN={MAX_LEN}  BATCH_SIZE={BATCH_SIZE}  num_workers={_nw}')


## Step 7 — Load Bio_ClinicalBERT


In [ ]:
print(f'Loading {MODEL_NAME}...')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    ignore_mismatched_sizes=True,
)
model = model.to(DEVICE)
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')


## Step 8 — Training helpers

`make_criterion` applies `NEURO_WEIGHT_BOOST` to neurological disease weights
after computing base inverse-frequency weights, then re-normalises.
This makes the training loss equally sensitive to neurological and
vector-borne misclassification errors.


In [ ]:
def run_epoch(model, dl, criterion, optimizer=None, scheduler=None, phase='train'):
    """Run one epoch. optimizer=None -> eval mode."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    first_step_loss = None
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for step, batch in enumerate(dl):
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            lbls = batch['label'].to(DEVICE)
            out  = model(input_ids=ids, attention_mask=mask)
            loss = criterion(out.logits, lbls)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                if scheduler:
                    scheduler.step()
            total_loss += loss.item()
            preds       = torch.argmax(out.logits, dim=1)
            correct    += (preds == lbls).sum().item()
            total      += lbls.size(0)
            if step == 0:
                first_step_loss = loss.item()
                if is_train:
                    print(f'  [{phase}] Step 0 loss={loss.item():.4f}  '
                          f'grad_norm={grad_norm:.4f}  '
                          f'(random baseline={RANDOM_BASELINE_LOSS:.3f})')
    return total_loss / len(dl), correct / total * 100, first_step_loss


def make_criterion(train_df):
    """
    Inverse-frequency class weights with neurological priority boost.

    Steps:
      1. Base weight = 1 / class_count  (inverse frequency)
      2. Normalise to sum = NUM_LABELS
      3. Multiply neurological weights by NEURO_WEIGHT_BOOST
      4. Re-normalise to sum = NUM_LABELS

    Result: neurological diseases have 30% higher loss contribution,
    matching their effective training priority to vector-borne diseases.
    """
    counts = train_df['label_id'].value_counts().sort_index()
    w = torch.tensor(
        [1.0 / counts[i] for i in range(NUM_LABELS)], dtype=torch.float
    )
    w = w / w.sum() * NUM_LABELS
    for name, lid in DISEASE_LABELS.items():
        if name in NEURO_DISEASES:
            w[lid] = w[lid] * NEURO_WEIGHT_BOOST
    w = w / w.sum() * NUM_LABELS   # re-normalise after boost
    w = w.to(DEVICE)
    print('Class weights (after neurological priority boost):')
    for i in range(NUM_LABELS):
        tag = '[neuro ]' if LABEL_NAMES[i] in NEURO_DISEASES else '[vector]'
        print(f'  {tag} {LABEL_NAMES[i]:25s}: {w[i].item():.4f}')
    return nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)


criterion = make_criterion(train_df)
print('Criterion ready.')


## Step 9 — Phase 1: Train classifier head only

Freeze all BERT encoder layers. Train only the classification head.
Fast, stable warmup before full fine-tuning.


In [ ]:
for name, param in model.named_parameters():
    if 'classifier' not in name:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 1: {trainable:,} trainable params (classifier head only)')

optimizer_p1   = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR_HEAD, weight_decay=0.01,
)
total_steps_p1 = len(train_dl) * PHASE1_EPOCHS
scheduler_p1   = get_linear_schedule_with_warmup(
    optimizer_p1,
    num_warmup_steps=int(0.1 * total_steps_p1),
    num_training_steps=total_steps_p1,
)

p1_train_losses, p1_train_accs = [], []
p1_val_losses,   p1_val_accs   = [], []

print(f'\n=== Phase 1: Classifier warmup ({PHASE1_EPOCHS} epochs) ===')
for epoch in range(1, PHASE1_EPOCHS + 1):
    tr_loss, tr_acc, step0 = run_epoch(
        model, train_dl, criterion, optimizer_p1, scheduler_p1,
        phase=f'P1-Ep{epoch}',
    )
    if epoch == 1 and step0 is not None and step0 >= RANDOM_BASELINE_LOSS:
        raise RuntimeError(
            f'Step 0 loss={step0:.4f} >= random baseline={RANDOM_BASELINE_LOSS:.3f}.\n'
            'Training is not converging. Check GPU and restart.'
        )
    vl_loss, vl_acc, _ = run_epoch(model, val_dl, criterion, phase=f'P1-Val{epoch}')
    p1_train_losses.append(tr_loss)
    p1_train_accs.append(tr_acc)
    p1_val_losses.append(vl_loss)
    p1_val_accs.append(vl_acc)
    print(f'Phase1 Ep {epoch}/{PHASE1_EPOCHS} '
          f'| Train loss={tr_loss:.4f} acc={tr_acc:.1f}% '
          f'| Val loss={vl_loss:.4f} acc={vl_acc:.1f}%')
    torch.save(model.state_dict(), f'checkpoint_p1_ep{epoch}.pt')

print('Phase 1 complete.')


## Step 10 — Phase 2: Full fine-tuning (all layers unfrozen)


In [ ]:
for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 2: {trainable:,} trainable params (all layers)')

optimizer_p2   = torch.optim.AdamW(
    model.parameters(), lr=LR_FULL, weight_decay=0.01,
)
total_steps_p2 = len(train_dl) * PHASE2_EPOCHS
scheduler_p2   = get_linear_schedule_with_warmup(
    optimizer_p2,
    num_warmup_steps=int(0.06 * total_steps_p2),
    num_training_steps=total_steps_p2,
)

p2_train_losses, p2_train_accs = [], []
p2_val_losses,   p2_val_accs   = [], []
best_val_loss = float('inf')
best_epoch    = 0
patience_ctr  = 0

print(f'\n=== Phase 2: Full fine-tuning ({PHASE2_EPOCHS} epochs) ===')
for epoch in range(1, PHASE2_EPOCHS + 1):
    tr_loss, tr_acc, _ = run_epoch(
        model, train_dl, criterion, optimizer_p2, scheduler_p2,
        phase=f'P2-Ep{epoch}',
    )
    vl_loss, vl_acc, _ = run_epoch(model, val_dl, criterion, phase=f'P2-Val{epoch}')
    p2_train_losses.append(tr_loss)
    p2_train_accs.append(tr_acc)
    p2_val_losses.append(vl_loss)
    p2_val_accs.append(vl_acc)
    print(f'Phase2 Ep {epoch}/{PHASE2_EPOCHS} '
          f'| Train loss={tr_loss:.4f} acc={tr_acc:.1f}% '
          f'| Val loss={vl_loss:.4f} acc={vl_acc:.1f}%')
    torch.save(model.state_dict(), f'checkpoint_p2_ep{epoch}.pt')
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        best_epoch    = epoch
        patience_ctr  = 0
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'  Best saved (val_loss={vl_loss:.4f})')
    else:
        patience_ctr += 1
        print(f'  No improvement ({patience_ctr}/{PATIENCE})')
        if patience_ctr >= PATIENCE:
            print(f'Early stopping at phase2 epoch {epoch}.')
            break

# Load best weights -- in this cell so it cannot be skipped
model.load_state_dict(torch.load('best_model.pt'))
TRAINING_COMPLETE = True
print(f'\nBest model (phase2 epoch {best_epoch}) loaded. Training complete.')


## Step 11 — Training sanity check


In [ ]:
assert TRAINING_COMPLETE, 'Training did not complete! Run Steps 9-10 first.'

from collections import Counter

model.eval()
quick_preds = []
with torch.no_grad():
    for batch in val_dl:
        out = model(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE),
        )
        quick_preds.extend(torch.argmax(out.logits, dim=1).cpu().tolist())

unique_classes = len(set(quick_preds))
dist = Counter(quick_preds)

print(f'Unique classes predicted on val set: {unique_classes} / {NUM_LABELS}')
print('\nPrediction distribution on val set:')
print('  --- Neurological ---')
for d in NEURO_DISEASES:
    lid = DISEASE_LABELS[d]
    bar = chr(9608) * (dist.get(lid, 0) // 3)
    print(f'  {d:25s}: {dist.get(lid, 0):4d}  {bar}')
print('  --- Vector-borne ---')
for d in VECTOR_DISEASES:
    lid = DISEASE_LABELS[d]
    bar = chr(9608) * (dist.get(lid, 0) // 3)
    print(f'  {d:25s}: {dist.get(lid, 0):4d}  {bar}')

if unique_classes < NUM_LABELS:
    missing = set(range(NUM_LABELS)) - set(quick_preds)
    print(f'WARNING: never predicted: {[LABEL_NAMES[m] for m in missing]}')
else:
    print('\nAll 12 disease classes predicted. No class collapse detected.')


## Step 12 — Evaluate on held-out test set


In [ ]:
assert TRAINING_COMPLETE, 'Training did not complete! Run Steps 9-10 first.'

model.eval()
all_preds, all_labels_list = [], []
with torch.no_grad():
    for batch in test_dl:
        out = model(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE),
        )
        all_preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
        all_labels_list.extend(batch['label'].numpy())

acc = accuracy_score(all_labels_list, all_preds)
f1m = f1_score(all_labels_list, all_preds, average='macro')
f1w = f1_score(all_labels_list, all_preds, average='weighted')
label_list = [LABEL_NAMES[i] for i in range(NUM_LABELS)]

print('=' * 58)
print('HealthAI v3 -- Test Set Results')
print('=' * 58)
print(f'Accuracy   : {acc*100:.2f}%')
print(f'F1 macro   : {f1m:.4f}')
print(f'F1 weighted: {f1w:.4f}')
print()
print(classification_report(all_labels_list, all_preds, target_names=label_list))


## Step 13 — Confusion matrix & training curves


In [ ]:
cm = confusion_matrix(all_labels_list, all_preds)
plt.figure(figsize=(14, 11))
sns.heatmap(
    cm, annot=True, fmt='d',
    xticklabels=label_list, yticklabels=label_list, cmap='Blues',
)
plt.title('HealthAI v3 -- Confusion Matrix', fontsize=13, fontweight='bold')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

all_train_losses = p1_train_losses + p2_train_losses
all_val_losses   = p1_val_losses   + p2_val_losses
all_train_accs   = p1_train_accs   + p2_train_accs
all_val_accs     = p1_val_accs     + p2_val_accs
epochs_ran       = len(all_train_losses)
phase_boundary   = PHASE1_EPOCHS + 0.5

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
x = range(1, epochs_ran + 1)
axes[0].plot(x, all_train_losses, 'o-', color='firebrick',  label='Train', linewidth=2)
axes[0].plot(x, all_val_losses,   's--', color='steelblue', label='Val',   linewidth=2)
axes[0].axvline(phase_boundary, color='gray', linestyle=':', label='Unfreeze')
axes[0].set_title('Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(x, all_train_accs, 'o-', color='firebrick',  label='Train', linewidth=2)
axes[1].plot(x, all_val_accs,   's--', color='steelblue', label='Val',   linewidth=2)
axes[1].axvline(phase_boundary, color='gray', linestyle=':', label='Unfreeze')
axes[1].set_title('Accuracy (%)', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.suptitle(
    'HealthAI v3 Training Curves (Phase1=head | Phase2=full)',
    fontsize=12, fontweight='bold',
)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Charts saved.')


## Step 14 — Live prediction demo

Three-layer correction stack active in `predict()`:
1. BERT softmax
2. TF-IDF cosine blend (with neurological boost when ambiguous)
3. Dengue specificity gate (hard penalty when no clinical markers)


In [ ]:
assert TRAINING_COMPLETE, 'Training did not complete! Run Steps 9-10 first.'


def has_dengue_markers(text):
    """Return True if text contains at least one dengue-specific clinical marker."""
    t = text.lower()
    return any(m in t for m in DENGUE_SPECIFIC_MARKERS)


def predict(text, use_tfidf=True):
    """
    Predict disease from free-text symptom description.

    Layer 1 - BERT softmax: primary signal from Bio_ClinicalBERT.

    Layer 2 - TF-IDF blend: triggered when:
      (a) BERT top-prob < AMBIGUITY_THRESHOLD  (model uncertain)
      (b) BERT predicts dengue AND top-prob < DENGUE_OVERRIDE_CAP
    Neurological cosine scores boosted by NEURO_TFIDF_BOOST in ambiguous cases.

    Layer 3 - Dengue gate: if dengue wins but no clinical markers present,
    multiply dengue prob by DENGUE_GATE_PENALTY so next disease surfaces.

    Returns
    -------
    disease    : str
    confidence : float (%)
    top3       : list of (disease, pct)
    blend_used : bool
    gate_used  : bool
    """
    model.eval()
    enc = tokenizer(
        str(text),
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt',
    )
    with torch.no_grad():
        out = model(
            input_ids=enc['input_ids'].to(DEVICE),
            attention_mask=enc['attention_mask'].to(DEVICE),
        )

    bert_probs = torch.softmax(out.logits, dim=1).cpu().numpy()[0]
    top_conf   = float(bert_probs.max())
    bert_pred  = int(bert_probs.argmax())
    dengue_id  = DISEASE_LABELS['dengue']
    blend_used = False
    gate_used  = False

    # Layer 2: TF-IDF blend
    trigger_ambig  = top_conf < AMBIGUITY_THRESHOLD
    trigger_dengue = (bert_pred == dengue_id and top_conf < DENGUE_OVERRIDE_CAP)

    if use_tfidf and (trigger_ambig or trigger_dengue):
        tfidf_scores  = tfidf_profiler.score(text, boost_neuro=True)
        dynamic_alpha = min(
            TFIDF_BLEND_ALPHA,
            (top_conf / AMBIGUITY_THRESHOLD) * TFIDF_BLEND_ALPHA,
        )
        probs  = dynamic_alpha * bert_probs + (1.0 - dynamic_alpha) * tfidf_scores
        probs /= probs.sum()
        blend_used = True
    else:
        probs = bert_probs.copy()

    # Layer 3: Dengue specificity gate
    if DENGUE_MARKER_GATE and int(probs.argmax()) == dengue_id:
        if not has_dengue_markers(text):
            probs[dengue_id] *= DENGUE_GATE_PENALTY
            probs /= probs.sum()
            gate_used = True

    pred = int(probs.argmax())
    top3 = [
        (LABEL_NAMES[int(i)], round(float(probs[i]) * 100, 1))
        for i in probs.argsort()[-3:][::-1]
    ]
    return LABEL_NAMES[pred], round(float(probs[pred]) * 100, 1), top3, blend_used, gate_used


# -- Clinical test cases (all 12 diseases with specific markers)
test_cases = [
    ('High fever, retro-orbital pain, rash, myalgia. NS1 antigen positive. Platelet 42000. Leucopenia.',
     'dengue'),
    ('Severe unilateral throbbing headache, photophobia, nausea. No fever. Normal CBC.',
     'migraine'),
    ('High fever, rigor, sweating. Blood smear P. vivax trophozoites. RDT pLDH positive.',
     'malaria'),
    ('Fever 3 weeks, eschar on axilla, lymphadenopathy. Weil-Felix OXK 1:160. IgM ELISA positive.',
     'scrub_typhus'),
    ('Prolonged fever, massive splenomegaly, weight loss. rK39 positive. Haemoglobin 7.2.',
     'kala_azar'),
    ('Breakthrough seizures, EEG spike-wave, on levetiracetam. AED levels subtherapeutic.',
     'epilepsy'),
    ('Progressive memory loss, confusion, wandering, age 74. MMSE 18. Hippocampal atrophy.',
     'dementia'),
    ('Sudden right-sided weakness, aphasia, 2 hours ago. CT no haemorrhage. MRI DWI infarct.',
     'stroke'),
    ('Resting tremor right hand, bradykinesia, rigidity. DaT scan reduced. Levodopa response.',
     'parkinsons'),
    ('Burning feet, numbness, HbA1c 9.2%. Nerve conduction reduced. 10g monofilament positive.',
     'diabetic_neuropathy'),
    ('Polyarthralgia, joint swelling, fever. IgM ELISA chikungunya positive.',
     'chikungunya'),
    ('Fever, altered consciousness, neck stiffness. CSF IgM JE positive. Thalamic T2 changes.',
     'japanese_encephalitis'),
]

# -- Ambiguous / generic cases (previously all predicted as dengue)
ambiguous_cases = [
    ('Fever and headache.',
     'ambiguous'),
    ('High fever, severe headache, muscle aches, fatigue.',
     'ambiguous'),
    ('Fever with joint pain and rash.',
     'ambiguous'),
    ('Cyclic high fever with rigors and sweating every 48 hours.',
     'malaria'),
    ('Persistent fever 3 weeks, weight loss, enlarged spleen.',
     'kala_azar'),
    ('Fever with eschar, swollen lymph nodes, rural scrub habitat.',
     'scrub_typhus'),
    ('Severe unilateral headache, vomiting, photophobia, no fever.',
     'migraine'),
    ('Confusion, memory problems, personality changes, age 70.',
     'dementia'),
    ('Sudden arm weakness, slurred speech, 1 hour ago.',
     'stroke'),
]

# -- Run clinical cases
print('HealthAI v3 -- Clinical Test Cases')
print('=' * 72)
correct = 0
predicted_classes = set()
for text, expected in test_cases:
    disease, conf, top3, blended, gated = predict(text)
    predicted_classes.add(disease)
    tick = 'CORRECT' if disease == expected else 'WRONG'
    if disease == expected:
        correct += 1
    notes = []
    if blended: notes.append('TF-IDF blend')
    if gated:   notes.append('dengue gate')
    note_str = '  [' + ', '.join(notes) + ']' if notes else ''
    print(f'Input    : {text[:72]}...')
    print(f'Predicted: {disease.upper()} ({conf}%) [{tick}]  Expected: {expected}{note_str}')
    print(f'Top 3    : {top3}')
    print()

print(f'Clinical accuracy  : {correct}/{len(test_cases)} ({correct/len(test_cases)*100:.0f}%)')
print(f'Unique classes     : {len(predicted_classes)}/{NUM_LABELS}')

# -- Run ambiguous cases
print()
print('HealthAI v3 -- Ambiguous Symptom Test')
print('=' * 72)
print('Goal: no dengue on generic inputs, neuro diseases surface correctly.\n')
dengue_count = 0
for text, note in ambiguous_cases:
    disease, conf, top3, blended, gated = predict(text)
    if disease == 'dengue':
        dengue_count += 1
    notes = []
    if blended: notes.append('TF-IDF blend')
    if gated:   notes.append('dengue gate')
    note_str = '  [' + ', '.join(notes) + ']' if notes else ''
    print(f'Input    : {text}')
    print(f'Expected : {note}')
    print(f'Predicted: {disease.upper()} ({conf}%){note_str}')
    print(f'Top 3    : {top3}')
    print()

print(f'Dengue in ambiguous set: {dengue_count}/{len(ambiguous_cases)}')
if dengue_count == 0:
    print('Dengue gate working perfectly -- no dengue on generic inputs.')
elif dengue_count <= 2:
    print('Good -- ambiguous inputs spread across diseases.')
else:
    print('Try lowering DENGUE_GATE_PENALTY or DENGUE_OVERRIDE_CAP.')


## Step 15 — Save & package for download


In [ ]:
assert TRAINING_COMPLETE, 'Training did not complete! Run Steps 9-10 first.'
import zipfile, pickle
from datetime import datetime

SAVE_DIR = 'healthai_classifier_v3'
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

with open(f'{SAVE_DIR}/tfidf_profiler.pkl', 'wb') as f:
    pickle.dump(tfidf_profiler, f)

with open(f'{SAVE_DIR}/label_map.json', 'w') as f:
    json.dump(LABEL_NAMES, f, indent=2)

with open(f'{SAVE_DIR}/model_info.json', 'w') as f:
    json.dump({
        'version':              'v3.0-neuro-equal-priority',
        'base_model':           MODEL_NAME,
        'train_samples':        len(train_df),
        'val_samples':          len(val_df),
        'test_samples':         len(test_df),
        'accuracy':             round(acc * 100, 2),
        'f1_macro':             round(f1m, 4),
        'f1_weighted':          round(f1w, 4),
        'best_p2_epoch':        best_epoch,
        'neuro_diseases':       NEURO_DISEASES,
        'vector_diseases':      VECTOR_DISEASES,
        'neuro_weight_boost':   NEURO_WEIGHT_BOOST,
        'neuro_tfidf_boost':    NEURO_TFIDF_BOOST,
        'dengue_marker_gate':   DENGUE_MARKER_GATE,
        'dengue_gate_penalty':  DENGUE_GATE_PENALTY,
        'tfidf_blend_alpha':    TFIDF_BLEND_ALPHA,
        'ambiguity_threshold':  AMBIGUITY_THRESHOLD,
        'dengue_override_cap':  DENGUE_OVERRIDE_CAP,
        'trained':              datetime.now().isoformat(),
    }, f, indent=2)

ZIP = 'healthai_model_v3.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir(SAVE_DIR):
        zf.write(f'{SAVE_DIR}/{fn}', fn)
    for fn in ['confusion_matrix.png', 'training_curves.png', 'best_model.pt']:
        if os.path.exists(fn):
            zf.write(fn, fn)

size_mb = os.path.getsize(ZIP) / 1e6
print(f'Saved: {ZIP}  ({size_mb:.1f} MB)')
print('Download: Files panel -> right-click -> Download')
print()
print('=' * 55)
print('HEALTHAI v3 -- TRAINING COMPLETE')
print('=' * 55)
print(f'Accuracy  : {acc*100:.2f}%')
print(f'F1 macro  : {f1m:.4f}')
print(f'Best epoch: Phase2 epoch {best_epoch}')
print('Neurological priority boost : ACTIVE')
print('Dengue specificity gate     : ACTIVE')
print('=' * 55)
